In [1]:
import pandas as pd
import requests
import json
import os
print("libraries loaded successfully")

libraries loaded successfully


In [2]:
url = "https://api.mfapi.in/mf/125497"
response = requests.get(url)
data = response.json()
print("Status code:", response.status_code)
print("Keys in response:",data.keys())
print("Fund info:",data['meta'])

Status code: 200
Keys in response: dict_keys(['meta', 'data', 'status'])
Fund info: {'fund_house': 'SBI Mutual Fund', 'scheme_type': 'Open Ended Schemes', 'scheme_category': 'Equity Scheme - Small Cap Fund', 'scheme_code': 125497, 'scheme_name': 'SBI Small Cap Fund - Direct Plan - Growth', 'isin_growth': 'INF200K01T51', 'isin_div_reinvestment': None}


In [3]:
print("Number of NAV records:", len(data['data']))
print("\n First 3 records:")
print(data['data'][:3])
print("\nLast 3 records:")
print(data['data'][-3:])

Number of NAV records: 3107

 First 3 records:
[{'date': '23-06-2026', 'nav': '203.54430'}, {'date': '22-06-2026', 'nav': '204.12630'}, {'date': '19-06-2026', 'nav': '202.07610'}]

Last 3 records:
[{'date': '20-11-2013', 'nav': '12.95490'}, {'date': '19-11-2013', 'nav': '13.10680'}, {'date': '18-11-2013', 'nav': '13.08940'}]


In [4]:
df = pd.DataFrame(data['data'])
print("Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())

Shape: (3107, 2)

Data Types:
date    str
nav     str
dtype: object

First 5 rows:
         date        nav
0  23-06-2026  203.54430
1  22-06-2026  204.12630
2  19-06-2026  202.07610
3  18-06-2026  200.95650
4  17-06-2026  199.83020


In [6]:
df['nav'] = pd.to_numeric(df['nav'])
df['date'] = pd.to_datetime(df['date'],format='%d-%m-%Y')
print("Fixed data types:")
print(df.dtypes)
print("\nFirst 5 rows:")
print(df.head())

Fixed data types:
date    datetime64[us]
nav            float64
dtype: object

First 5 rows:
        date       nav
0 2026-06-23  203.5443
1 2026-06-22  204.1263
2 2026-06-19  202.0761
3 2026-06-18  200.9565
4 2026-06-17  199.8302


In [7]:
# Save to data/raw/
os.makedirs('data/raw', exist_ok=True)

df.to_csv('data/raw/sbi_small_cap_nav.csv', index=False)

print("Saved! Let's verify:")
df_check = pd.read_csv('data/raw/sbi_small_cap_nav.csv')
print(df_check.head())

Saved! Let's verify:
         date       nav
0  2026-06-23  203.5443
1  2026-06-22  204.1263
2  2026-06-19  202.0761
3  2026-06-18  200.9565
4  2026-06-17  199.8302


In [10]:
def fetch_and_save_nav(scheme_code, filename):
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    response = requests.get(url)
    fund_data = response.json()  # renamed to fund_data to avoid conflict
    
    # Get fund name from meta
    fund_name = fund_data['meta']['scheme_name']
    
    # Convert to DataFrame and fix types
    df = pd.DataFrame(fund_data['data'])
    df['nav'] = pd.to_numeric(df['nav'])
    df['date'] = pd.to_datetime(df['date'], format='%d-%m-%Y')
    
    # Save to CSV
    df.to_csv(f'data/raw/{filename}.csv', index=False)
    
    print(f"✅ Saved {fund_name} — {len(df)} records")
    return df

# Fetch all 6 funds
funds = {
    125497: 'sbi_small_cap',
    119551: 'sbi_bluechip',
    120503: 'icici_bluechip',
    118632: 'nippon_large_cap',
    119092: 'axis_bluechip',
    120841: 'kotak_bluechip'
}

for code, name in funds.items():
    fetch_and_save_nav(code, name)

✅ Saved SBI Small Cap Fund - Direct Plan - Growth — 3107 records
✅ Saved Aditya Birla Sun Life Banking & PSU Debt Fund  - DIRECT - IDCW — 3252 records
✅ Saved Axis ELSS Tax Saver Fund - Direct Plan - Growth Option — 3323 records
✅ Saved Nippon India Large Cap Fund - Direct Plan Growth Plan - Growth Option — 3314 records
✅ Saved HDFC Money Market Fund - Growth Option - Direct Plan — 3581 records
✅ Saved quant Mid Cap Fund - Growth Option - Direct Plan — 3317 records


In [11]:
files = os.listdir('data/raw')
print("Files in data/raw:")
for f in files:
    print(f" -{f}")

Files in data/raw:
 -axis_bluechip.csv
 -icici_bluechip.csv
 -kotak_bluechip.csv
 -nippon_large_cap.csv
 -sbi_bluechip.csv
 -sbi_small_cap.csv
 -sbi_small_cap_nav.csv


In [12]:
# Load and explore all 6 funds
csv_files = {
    'SBI Small Cap': 'data/raw/sbi_small_cap.csv',
    'SBI Bluechip': 'data/raw/sbi_bluechip.csv',
    'ICICI Bluechip': 'data/raw/icici_bluechip.csv',
    'Nippon Large Cap': 'data/raw/nippon_large_cap.csv',
    'Axis Bluechip': 'data/raw/axis_bluechip.csv',
    'Kotak Bluechip': 'data/raw/kotak_bluechip.csv',
}

for fund_name, filepath in csv_files.items():
    df = pd.read_csv(filepath)
    print(f"\n{'='*50}")
    print(f"Fund: {fund_name}")
    print(f"Shape: {df.shape}")
    print(f"Dtypes:\n{df.dtypes}")
    print(f"Head:\n{df.head(3)}")
    print(f"Missing values:\n{df.isnull().sum()}")


Fund: SBI Small Cap
Shape: (3107, 2)
Dtypes:
date        str
nav     float64
dtype: object
Head:
         date       nav
0  2026-06-23  203.5443
1  2026-06-22  204.1263
2  2026-06-19  202.0761
Missing values:
date    0
nav     0
dtype: int64

Fund: SBI Bluechip
Shape: (3252, 2)
Dtypes:
date        str
nav     float64
dtype: object
Head:
         date       nav
0  2026-06-23  106.0410
1  2026-06-22  105.9850
2  2026-06-19  105.9219
Missing values:
date    0
nav     0
dtype: int64

Fund: ICICI Bluechip
Shape: (3323, 2)
Dtypes:
date        str
nav     float64
dtype: object
Head:
         date       nav
0  2026-06-23  107.5421
1  2026-06-22  108.4681
2  2026-06-19  107.9636
Missing values:
date    0
nav     0
dtype: int64

Fund: Nippon Large Cap
Shape: (3314, 2)
Dtypes:
date        str
nav     float64
dtype: object
Head:
         date       nav
0  2026-06-23  100.4605
1  2026-06-22  101.3654
2  2026-06-19  100.7824
Missing values:
date    0
nav     0
dtype: int64

Fund: Axis Bluechip
Shap

In [13]:
# Fetch AMFI fund master data
url = "https://api.mfapi.in/mf"
response = requests.get(url)
fund_master = response.json()

print("Type:", type(fund_master))
print("Number of funds:", len(fund_master))
print("\nFirst fund:")
print(fund_master[0])
print("\nSecond fund:")
print(fund_master[1])

Type: <class 'list'>
Number of funds: 37647

First fund:
{'schemeCode': 100027, 'schemeName': 'Grindlays Super Saver Income Fund-GSSIF-Half Yearly Dividend', 'isinGrowth': None, 'isinDivReinvestment': None}

Second fund:
{'schemeCode': 100028, 'schemeName': 'Grindlays Super Saver Income Fund-GSSIF-Quaterly Dividend', 'isinGrowth': None, 'isinDivReinvestment': None}


In [14]:
# Convert to DataFrame
df_master = pd.DataFrame(fund_master)

print("Shape:", df_master.shape)
print("\nDtypes:")
print(df_master.dtypes)
print("\nFirst 5 rows:")
print(df_master.head())
print("\nMissing values:")
print(df_master.isnull().sum())

Shape: (37647, 4)

Dtypes:
schemeCode             int64
schemeName               str
isinGrowth               str
isinDivReinvestment      str
dtype: object

First 5 rows:
   schemeCode                                         schemeName isinGrowth  \
0      100027  Grindlays Super Saver Income Fund-GSSIF-Half Y...        NaN   
1      100028  Grindlays Super Saver Income Fund-GSSIF-Quater...        NaN   
2      100029     Grindlays Super Saver Income Fund-GSSIF-Growth        NaN   
3      100030  Grindlays Super Saver Income Fund-GSSIF-Annual...        NaN   
4      100031  Grindlays Super Saver Income Fund-GSSIF - ST-D...        NaN   

  isinDivReinvestment  
0                 NaN  
1                 NaN  
2                 NaN  
3                 NaN  
4                 NaN  

Missing values:
schemeCode                 0
schemeName                 0
isinGrowth             29220
isinDivReinvestment    33506
dtype: int64


In [15]:
# How many unique fund houses are there?
# Fund house name is embedded in the scheme name
# Let's look at some patterns
print("Total funds:", len(df_master))

# Search for some known fund houses
keywords = ['SBI', 'HDFC', 'ICICI', 'Axis', 'Nippon', 'Kotak']

for keyword in keywords:
    count = df_master['schemeName'].str.contains(keyword, case=False).sum()
    print(f"{keyword}: {count} funds")

Total funds: 37647
SBI: 1934 funds
HDFC: 3301 funds
ICICI: 3220 funds
Axis: 789 funds
Nippon: 1111 funds
Kotak: 1525 funds


In [17]:
# Save fund master to CSV
df_master.to_csv('data/raw/fund_master.csv', index=False)
print("✅ Fund master saved!")

# Data Quality Summary
print("\n" + "="*50)
print("DATA QUALITY SUMMARY — DAY 1")
print("="*50)

print("\n📁 FILES SAVED IN data/raw/:")
for f in os.listdir('data/raw'):
    size = os.path.getsize(f'data/raw/{f}')
    print(f"  - {f} ({size/1024:.1f} KB)")

print("\n📊 NAV DATA QUALITY:")
nav_files = {
    'SBI Small Cap': 'data/raw/sbi_small_cap.csv',
    'SBI Bluechip': 'data/raw/sbi_bluechip.csv',
    'ICICI Bluechip': 'data/raw/icici_bluechip.csv',
    'Nippon Large Cap': 'data/raw/nippon_large_cap.csv',
    'Axis Bluechip': 'data/raw/axis_bluechip.csv',
    'Kotak Bluechip': 'data/raw/kotak_bluechip.csv',
}
for name, path in nav_files.items():
    df_temp = pd.read_csv(path)
    missing = df_temp.isnull().sum().sum()
    print(f"  - {name}: {len(df_temp)} records, {missing} missing values")

print("\n📋 FUND MASTER QUALITY:")
print(f"  - Total funds: {len(df_master)}")
print(f"  - Missing ISIN Growth: {df_master['isinGrowth'].isnull().sum()} ({df_master['isinGrowth'].isnull().mean()*100:.1f}%)")
print(f"  - Missing ISIN Div: {df_master['isinDivReinvestment'].isnull().sum()} ({df_master['isinDivReinvestment'].isnull().mean()*100:.1f}%)")

print("\n⚠️  ANOMALIES FOUND:")
print("  - Scheme codes in task did not match expected fund names")
print("  - ISIN data missing for 78-89% of funds (likely legacy/inactive funds)")
print("  - Date column reverts to string after CSV save/load (requires parse_dates on read)")

✅ Fund master saved!

DATA QUALITY SUMMARY — DAY 1

📁 FILES SAVED IN data/raw/:
  - axis_bluechip.csv (75.0 KB)
  - fund_master.csv (2939.8 KB)
  - icici_bluechip.csv (65.0 KB)
  - kotak_bluechip.csv (65.7 KB)
  - nippon_large_cap.csv (64.6 KB)
  - sbi_bluechip.csv (66.3 KB)
  - sbi_small_cap.csv (61.6 KB)
  - sbi_small_cap_nav.csv (61.6 KB)

📊 NAV DATA QUALITY:
  - SBI Small Cap: 3107 records, 0 missing values
  - SBI Bluechip: 3252 records, 0 missing values
  - ICICI Bluechip: 3323 records, 0 missing values
  - Nippon Large Cap: 3314 records, 0 missing values
  - Axis Bluechip: 3581 records, 0 missing values
  - Kotak Bluechip: 3317 records, 0 missing values

📋 FUND MASTER QUALITY:
  - Total funds: 37647
  - Missing ISIN Growth: 29220 (77.6%)
  - Missing ISIN Div: 33506 (89.0%)

⚠️  ANOMALIES FOUND:
  - Scheme codes in task did not match expected fund names
  - ISIN data missing for 78-89% of funds (likely legacy/inactive funds)
  - Date column reverts to string after CSV save/load (